In [153]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

## Loading Dataset

In [154]:
df = pd.read_csv("xray_dataset/chest_xray.csv")
df['Label'] = df['Finding Labels'].apply(lambda x: 0 if x == 'No Finding' else 1)

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['Label'])
val_df, test_df = train_test_split(temp_df, test_size=2/3, stratify=temp_df['Label'])

In [155]:
# print the distribution in the classes
print("Train set distribution:")
print(train_df['Label'].value_counts(normalize=True))
print("Validation set distribution:")
print(val_df['Label'].value_counts(normalize=True))
print("Test set distribution:")
print(test_df['Label'].value_counts(normalize=True))


Train set distribution:
Label
0    0.516224
1    0.483776
Name: proportion, dtype: float64
Validation set distribution:
Label
0    0.517241
1    0.482759
Name: proportion, dtype: float64
Test set distribution:
Label
0    0.515464
1    0.484536
Name: proportion, dtype: float64


In [156]:
class ChestXrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.df.iloc[idx]['Image Index'])
        image = Image.open(img_name).convert('RGB')
        label = self.df.iloc[idx]['Label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [157]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),  # Add normalization back in
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # Add slight rotation
    transforms.RandomAffine(0, translate=(0.05, 0.05))  # Add slight translation
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # No augmentations for validation/test
])

img_dir = "xray_dataset/images"  # Directory containing the images

# Update the datasets with proper transforms
train_data = ChestXrayDataset(train_df, img_dir, transform_train)
val_data = ChestXrayDataset(val_df, img_dir, transform_val)
test_data = ChestXrayDataset(test_df, img_dir, transform_val)

# Handle class imbalance (if necessary)
# Calculate weights for the sampler
# Handle class imbalance
class_counts = train_df['Label'].value_counts().sort_index()
weights = 1.0 / class_counts
sample_weights = weights[train_df['Label'].values]
# Convert pandas Series to numpy array or list
sample_weights = sample_weights.to_numpy()  # Convert to numpy array

sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Update DataLoader with the sampler
train_loader = DataLoader(train_data, batch_size=16, sampler=sampler)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)

In [158]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)  # (B, embed_dim, num_patches^(1/2), num_patches^(1/2))
        x = x.flatten(2)  # Flatten height & width
        x = x.transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
    
    def forward(self, x):
        x = x.permute(1, 0, 2)  # (Seq_len, Batch, Embed_dim) for MultiheadAttention
        attn_output, _ = self.attention(x, x, x)
        return attn_output.permute(1, 0, 2)  # Convert back

class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, mlp_dim=3072, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # Add residual connection
        x = x + self.mlp(self.ln2(x))  # Add residual connection
        return x

class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=384, num_heads=6, 
                 depth=8, num_classes=2, mlp_dim=1536, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        
        num_patches = (img_size // patch_size) ** 2
        # Fix positional embeddings size
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, embed_dim))
        
        # Initialize positional embeddings
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        
        self.dropout = nn.Dropout(dropout)
        self.encoder = nn.Sequential(
            *[TransformerEncoder(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)]
        )
        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, num_patches, embed_dim)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # Add class token
        x = torch.cat([cls_tokens, x], dim=1)  # (B, num_patches+1, embed_dim)
        x = x + self.pos_embed  # Add positional embedding
        x = self.dropout(x)
        x = self.encoder(x)
        x = self.ln(x[:, 0])  # Take CLS token
        return self.head(x)

# Create model instance
vit = VisionTransformer()

# Test on random image batch
dummy_img = torch.randn(4, 3, 224, 224)
output = vit(dummy_img)
print(output.shape)  # Expected: (4, num_classes)

torch.Size([4, 2])


In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer(
    img_size=224,
    patch_size=16,
    in_channels=3,
    embed_dim=384,  # Reduced from 768
    num_heads=8,    # Reduced from 8
    depth=12,        # Reduced from 12
    mlp_dim=1536,   # Reduced from 3072
    dropout=0.1
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss()
# Add gradient clipping
max_grad_norm = 1.0

In [160]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def test_model(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend(labels.numpy())

    print("Test Accuracy:", accuracy_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("Classification Report:\n", classification_report(y_true, y_pred, target_names=["No Finding", "With Finding"]))


In [161]:
from tqdm import tqdm
train_losses, train_accs = [], []
val_losses, val_accs = [], []
num_epochs = 20
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    
    # Training
    train_loss, train_acc = train_one_epoch(
        model, tqdm(train_loader, desc="Training"), 
        optimizer, criterion, device
    )
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validation
    val_loss, val_acc = evaluate(
        model, tqdm(val_loader, desc="Validation"), 
        criterion, device
    )
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Update learning rate
    scheduler.step()
    
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

Epoch 1/20


Validation: 100%|██████████| 19/19 [00:05<00:00,  3.24it/s]


Train Loss: 0.7502 | Train Acc: 0.5147
Val Loss:   0.6946 | Val Acc:   0.5138
Learning Rate: 0.000098
Epoch 2/20


Validation: 100%|██████████| 19/19 [00:05<00:00,  3.19it/s]


Train Loss: 0.7047 | Train Acc: 0.5025
Val Loss:   0.6893 | Val Acc:   0.5000
Learning Rate: 0.000090
Epoch 3/20


...

Train Loss: 0.6772 | Train Acc: 0.5737
Val Loss:   0.6752 | Val Acc:   0.5138
Learning Rate: 0.000021
Epoch 14/20


Validation: 100%|██████████| 19/19 [00:05<00:00,  3.21it/s]


Train Loss: 0.6829 | Train Acc: 0.5536
Val Loss:   0.6720 | Val Acc:   0.5862
Learning Rate: 0.000035
Epoch 15/20


Training:  78%|███████▊  | 100/128 [00:47<00:13,  2.12it/s]


KeyboardInterrupt: 

In [162]:
test_model(model, test_loader, device)


Test Accuracy: 0.5532646048109966
Confusion Matrix:
 [[127 173]
 [ 87 195]]
Classification Report:
               precision    recall  f1-score   support

  No Finding       0.59      0.42      0.49       300
With Finding       0.53      0.69      0.60       282

    accuracy                           0.55       582
   macro avg       0.56      0.56      0.55       582
weighted avg       0.56      0.55      0.55       582

